# EXP001 division 深度诊断:预测 fork vs GT 分裂

In [ ]:
import os, subprocess, sys, glob, shutil
os.environ['POLARS_PREFER_PKG'] = '32'
wheel_dir = '/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels'
targets = []
for pat in ['tracksdata-*.whl','geff-*.whl','geff_spec-*.whl','zarr-3*.whl','numcodecs-0.15*.whl','blosc2-*.whl','donfig-*.whl','pyscipopt-*.whl','ilpy-*.whl','rustworkx-*.whl','polars-*.whl','polars_runtime_32-*.whl','bidict-*.whl','psygnal-*.whl','networkx-*.whl','pydantic-*.whl','pydantic_core-*.whl','annotated_types-*.whl','typing_inspection-*.whl','markdown_it_py-*.whl','pygments-*.whl','click-*.whl','cloudpickle-*.whl','fsspec-*.whl','partd-*.whl','locket-*.whl','toolz-*.whl','yaml-*.whl','ndindex-*.whl','msgpack-*.whl','numexpr-*.whl','deprecated-*.whl','wrapt-*.whl','rich-*.whl']:
    targets += sorted(glob.glob(wheel_dir + '/' + pat))
print('installing', len(targets), 'wheels')
for m in targets:
    r = subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',m],
                      capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        print('FAIL', m.split('/')[-1], (r.stderr or '')[-200:])
import tracksdata as td
print('deps ok')


In [ ]:
# materialize inference repo + weights, run predict on one division-rich embryo
import json, os, shutil
from pathlib import Path
ART = Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')
REPO = Path('/kaggle/working/tracking_repo')
if not (REPO/'scripts').exists():
    shutil.copytree(ART/'repo', REPO)
if not (REPO/'weights').exists():
    shutil.copytree(ART/'weights', REPO/'weights')
print('repo + weights ready; weight exists:', (REPO/'weights/unet_transformer/split_0/edge_predictor_best.pth').exists())
TRAIN_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
split_json = REPO/'diag_split.json'
split_json.write_text(json.dumps([{'split':0,'train':[],'test':['6bba_969618f6']}]))
cmd = [sys.executable, 'scripts/predict_unet_transformer.py',
       '--data-dir', str(TRAIN_DIR),
       '--splits', split_json.name,
       '--split', '0',
       '--weights', 'weights/unet_transformer/split_0/edge_predictor_best.pth',
       '--unet-batch-size', '4',
       '--det-threshold', '0.96875',
       '--ilp-edge-weight', '-1.0',
       '--ilp-appearance-weight', '0.0',
       '--ilp-disappearance-weight', '1.575',
       '--ilp-division-weight', '1.0',
       '--use-ilp']
print('running predict...')
import subprocess
r = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, 'PYTHONPATH': str(REPO/'src')}, capture_output=True, text=True, timeout=3600)
print('rc=', r.returncode)
print((r.stdout or '')[-1500:])
print((r.stderr or '')[-1500:])
out_geff = REPO/'predictions'/'unknown'/'unet_transformer'/'split_0'/'6bba_969618f6.geff'
print('geff exists:', out_geff.exists())
if not out_geff.exists():
    for f in (REPO/'predictions').rglob('*.geff'):
        print('found', f)


In [ ]:
# ---- analyze predicted forks vs GT divisions on 6bba_969618f6 ----
import numpy as np, pandas as pd
import tracksdata as td
from pathlib import Path
REPO = Path('/kaggle/working/tracking_repo')
TRAIN_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
SCALE = np.array([1.625, 0.40625, 0.40625])
def read_g(path):
    g = td.graph.IndexedRXGraph.from_geff(path)
    return g[0] if isinstance(g, tuple) else g
def frames(g):
    n = pd.DataFrame(g.node_attrs().to_dicts())
    e = pd.DataFrame(g.edge_attrs().to_dicts())
    return n, e
# GT
gn, ge = frames(read_g(TRAIN_DIR/'6bba_969618f6.geff'))
gsrc = ge.groupby('source_id').size()
gt_div = gsrc[gsrc==2].index.tolist()
print('GT divisions on this embryo:', gt_div)
# Pred
pf = REPO/'predictions'/'unknown'/'unet_transformer'/'split_0'/'6bba_969618f6.geff'
if not pf.exists():
    print('pred geff missing'); raise SystemExit
pn, pe = frames(read_g(pf))
psrc = pe.groupby('source_id').size()
pred_forks = psrc[psrc>=2].index.tolist()
print('pred forks (out>=2):', len(pred_forks))
# For each GT division, find nearest pred fork in space+time
gt_t = dict(zip(gn.node_id, gn.t))
gt_xyz = gn.set_index('node_id')[['z','y','x']]
pred_t = dict(zip(pn.node_id, pn.t))
pred_xyz = pn.set_index('node_id')[['z','y','x']]
import itertools
def dist_um(a, b):
    return float(np.linalg.norm((a-b)*SCALE))
print('\nGT division -> nearest pred forks (same t-1..t+1, sorted by dist):')
for gd in gt_div:
    gt_tm = gt_t[gd]
    kids = ge[ge.source_id==gd].target_id.tolist()
    print(f'\nGT div {gd} t={gt_tm} kids={kids}')
    cands = []
    for pfk in pred_forks:
        pt = pred_t[pfk]
        if abs(pt - gt_tm) <= 1:
            d = dist_um(pred_xyz.loc[pfk].values, gt_xyz.loc[gd].values)
            cands.append((d, pfk, pt))
    cands.sort()
    for d, pfk, pt in cands[:5]:
        pkids = pe[pe.source_id==pfk].target_id.tolist()
        pkids_t = [(k, pred_t[k]) for k in pkids]
        print(f'  pred fork {pfk} t={pt} dist={d:.2f}um kids={pkids_t}')
